# 02 · Inference Demo — Ask, Extract, Highlight

This notebook drives the model end-to-end through the project's single inference
facade, `DocumentPredictor`. On one DocVQA page we will:

1. **Answer** a natural-language question and read off the answer + confidence.
2. **Extract** structured fields as JSON (invoice/receipt-style schema).
3. **Highlight** the region of the page the answer was grounded to (OCR-based).

> **Requirements.** Needs the full dependencies (`torch`, `transformers`,
> `peft`, `Pillow`; Tesseract + `pytesseract` for highlighting). If
> `config.inference.adapter_path` points at a trained LoRA adapter it is loaded
> automatically; otherwise the predictor falls back to the **base model
> (zero-shot)** — everything below still runs, the answers are just un-tuned.


## Setup

In [ ]:
# --- Make the repo root importable (works whether run from notebooks/ or root)
import sys
from pathlib import Path

# Walk up until we find the repo marker (pyproject.toml); fall back to parent.
_here = Path.cwd()
_root = next(
    (p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists()),
    _here.parent,
)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Repo root on sys.path: {_root}")


In [ ]:
from configs import load_config
from utils.logging_utils import get_logger
from utils.device import get_device_info

logger = get_logger("nb.inference_demo")

config = load_config()
# A quick device readout so it is obvious whether we are on GPU/MPS/CPU — it sets
# expectations for the latency numbers reported further down.
info = get_device_info(config.device, config.model.torch_dtype)
print("Device :", info.device_name, f"({info.device})")
print("Dtype  :", info.dtype)
print("Adapter:", config.inference.adapter_path or "(none — base model, zero-shot)")


## Build the predictor

`DocumentPredictor.from_config` builds the `VisionDocModel` and, if an adapter
path is configured and exists, attaches the LoRA weights. It also owns the OCR
engine (for highlighting) and the field extractor. The first call is slow — the
multi-GB backbone is loaded onto the device here.


In [ ]:
from inference.predictor import DocumentPredictor

# One object serves every surface (API, Streamlit app, batch jobs). use_ocr=True
# enables answer -> bounding-box grounding when Tesseract is available; it
# degrades gracefully to "no regions" when it is not.
predictor = DocumentPredictor.from_config(config)
print("Predictor ready. OCR available:", predictor.use_ocr)


## Pick a document

Grab a single validation sample so we have a real page plus a known-good question and gold answer to sanity-check against. Swap in your own image path or `question` to try anything else.

In [ ]:
from preprocessing.datasets import load_samples
from utils.image_utils import load_image
import matplotlib.pyplot as plt

sample = load_samples(config, "validation", max_samples=1)[0]
image = load_image(sample.image)          # PIL.Image (RGB)
question = sample.question

print("Question   :", question)
print("Gold answer:", sample.answer, f"(all acceptable: {sample.answers})")

fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(image)
ax.axis("off")
ax.set_title("Input document", fontsize=12)
plt.show()


## 1 · Answer a question

`predictor.answer` (aliased as `.ask`) returns a `PredictionResult`: the answer text, a calibrated confidence in `[0, 1]`, wall-clock latency, and — when OCR finds the answer on the page — the highlighted regions.

In [ ]:
result = predictor.answer(image, question, highlight=True)

print(f"Answer     : {result.answer}")
print(f"Confidence : {result.confidence:.1%}")
print(f"Latency    : {result.latency_ms:.0f} ms")
print(f"Regions    : {len(result.regions)} box(es) grounded on the page")
print(f"Gold answer: {sample.answer}")


## Show the highlighted answer

When OCR localises the answer, `PredictionResult` carries a base64 PNG of the page with the region boxed. We decode it with `utils.base64_to_pil`; if grounding failed we fall back to the original upload.

In [ ]:
from utils.image_utils import base64_to_pil

# Prefer the highlighted render; fall back to the raw page if OCR found nothing.
if result.highlighted_image_b64:
    display_img = base64_to_pil(result.highlighted_image_b64)
    caption = "Answer highlighted (OCR-grounded region)"
else:
    display_img = image
    caption = "No region highlighted (OCR unavailable or answer not located)"

fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(display_img)
ax.axis("off")
ax.set_title(caption, fontsize=12)
plt.show()


## 2 · Structured field extraction

Document QA answers one question at a time; **field extraction** pulls a whole
schema at once and returns JSON. `predictor.extract` accepts either an explicit
`fields` list or a `doc_type` whose canonical field set lives in
`inference.extract.DOC_TYPE_FIELDS` (`invoice`, `receipt`, `id_card`, `form`).

The DocVQA page above is unlikely to *be* an invoice, so treat this as a
demonstration of the API and the JSON contract — on a real invoice/receipt the
values populate. Missing fields come back as `None` (schema-stable output).


In [ ]:
from inference.extract import DOC_TYPE_FIELDS
import json

print("Invoice schema:", DOC_TYPE_FIELDS["invoice"], "\n")

# doc_type="invoice" expands to the canonical invoice field set above.
extraction = predictor.extract(image, doc_type="invoice")

print("Confidence:", f"{extraction['confidence']:.1%}")
print("Fields    :")
print(json.dumps(extraction["fields"], indent=2, ensure_ascii=False))
print("\nRaw model text (for audit):")
print(extraction["raw"][:500])


In [ ]:
# You can also request an ad-hoc field list without naming a document type.
custom = predictor.extract(image, fields=["title", "date", "total"])
print(json.dumps(custom["fields"], indent=2, ensure_ascii=False))


## 3 · Region highlighting directly

`predictor.highlight_regions` is the standalone primitive behind the highlighted answer above: give it any image and an answer string and it returns the page with the located text boxed (unchanged if OCR is unavailable or the text is not found). Handy for overlaying an arbitrary target phrase.

In [ ]:
# Highlight wherever the gold answer text appears on the page.
highlighted = predictor.highlight_regions(image, sample.answer)

fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(highlighted)
ax.axis("off")
ax.set_title(f"highlight_regions(answer={sample.answer!r})", fontsize=11)
plt.show()


## Takeaways

- `DocumentPredictor` is the one object every surface (API, app, batch) uses —
  ask, extract, and highlight all flow through it with a single result shape.
- Confidence is a real sequence-probability proxy, not a constant — useful for
  routing low-confidence pages to human review.

**Next:** `03_evaluation_and_comparison.ipynb` quantifies all of this — scoring
the base model vs. a LoRA fine-tune on a validation slice and plotting the lift.
